In [2]:

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, random_split

# 固定中文配置
plt.rcParams["font.family"] = ["SimHei", "WenQuanYi Micro Hei", "Heiti TC"]
plt.rcParams["axes.unicode_minus"] = False

# 自动检测设备
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224 if device == "cuda" else 128
BATCH_SIZE = 32

# 复用你之前的训练/验证增强、数据集拆分逻辑
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])
transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# 复用你之前的拆分索引，保证训练/验证划分和之前一致
full_dataset = datasets.ImageFolder(root="D:/cv_learning/3_pytorch_corn_disease/corn_disease")
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_idx, val_idx = random_split(full_dataset, [train_size, val_size])

train_dataset = datasets.ImageFolder(root="D:/cv_learning/3_pytorch_corn_disease/corn_disease", transform=transform_train)
val_dataset = datasets.ImageFolder(root="D:/cv_learning/3_pytorch_corn_disease/corn_disease", transform=transform_val)
train_dataset = torch.utils.data.Subset(train_dataset, train_idx.indices)
val_dataset = torch.utils.data.Subset(val_dataset, val_idx.indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 规范写法：用weights参数加载预训练权重，旧pretrained写法已废弃
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
# 把模型移到对应设备
model = model.to(device)

# 1. 冻结前面所有骨干层：所有参数不更新梯度
for param in model.parameters():
    param.requires_grad = False

# 2. 拿到ResNet最后全连接层的输入维度（ResNet18固定为512）
fc_in_features = model.fc.in_features
# 3. 替换最后的全连接层，输出3类玉米病害，新层默认会更新梯度
model.fc = nn.Linear(fc_in_features, out_features=3).to(device)

# 定义损失、优化器，只把需要训练的分类头参数传入优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

print("新分类头是否可训练：", model.fc.weight.requires_grad)
print("骨干卷积层是否冻结：", model.conv1.weight.requires_grad)

# 跑1轮训练
model.train()
train_loss = 0.0
train_correct = 0
for images, labels in train_loader:
    optimizer.zero_grad()
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    train_correct += (predicted == labels).sum().item()

print(f"1轮训练完成，平均损失：{train_loss/len(train_loader):.3f}，训练准确率：{100*train_correct/len(train_dataset):.1f}%")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\xning/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


新分类头是否可训练： True
骨干卷积层是否冻结： False
1轮训练完成，平均损失：0.415，训练准确率：88.4%
